# 1.0 Importando pacotes

In [1]:
import geopandas as gpd
from pathlib import Path
import pandas as pd
import numpy as np
import os

# 2.0 Definindo caminhos

In [2]:
# Caminho da pasta de inputs
inputs_path = Path('./inputs').resolve()

# Caminho da pasta de outputs
outputs_path = Path('./outputs').resolve()

# Caminhos dos subsets
co_path = inputs_path / 'buffered_subset_co.parquet'
no2_path = inputs_path / 'buffered_subset_no2.parquet'
o3_path = inputs_path / 'buffered_subset_o3.parquet'
pm_path = inputs_path / 'buffered_subset_pm.parquet'
so2_path = inputs_path / 'buffered_subset_so2.parquet'


# 3.0 Carregando subsets

In [3]:
buffered_subset_co = gpd.read_parquet(co_path)
buffered_subset_no2 = gpd.read_parquet(no2_path)
buffered_subset_o3 = gpd.read_parquet(o3_path)
buffered_subset_pm = gpd.read_parquet(pm_path)
buffered_subset_so2 = gpd.read_parquet(so2_path)

# 4.0 Formatando outputs

In [4]:
# 1) estacoes_completa --------------------------------------------------------------
# Unindo poluentes em um gdf final completo
buffered_stations = pd.concat([buffered_subset_co,
                               buffered_subset_no2,
                               buffered_subset_o3,
                               buffered_subset_pm,
                               buffered_subset_so2])


# 2) rep_espacial -------------------------------------------------------------------
"""
Planilha original de estações de monitoramento + colunas [REP_ESPACIAL_NAME] e [REP_ESPACIAL]
    - REP_ESPACIAL_NAME: nome da classe de representatividade espacial (micro, meso, 
    bairro, urbana)
    - REP_ESPACIAL: tamanho do raio do buffer de representatividade em metros
"""
# Removendo colunas auxiliares
def drop_aux_cols(subset):
    return subset.drop(columns= (list(subset
                                      .filter(like='rep')
                                      .columns) +
                                 list(subset
                                      .filter(like='min')
                                      .columns) +
                                 list(subset
                                      .filter(like='max')
                                      .columns) +
                                 list(subset
                                      .filter(like='k')
                                      .columns) +
                                 list(subset
                                      .filter(like='micro')
                                      .columns) +
                                 list(subset
                                      .filter(like='meso')
                                      .columns) +
                                 list(subset
                                      .filter(like='bairro')
                                      .columns) +
                                 list(subset
                                      .filter(like='urb')
                                      .columns) +
                                 ['EPSG','distance_to_industry',
                                  'industry_geom']
                                 )
                       )
# Aplicando função
filtered_stations = drop_aux_cols(buffered_stations) 


# 5.0 Salvando outputs

In [5]:
# Salvando o GeoDataFrame com a indústria mais próxima de cada estação
buffered_stations.to_parquet(outputs_path / 'estacoes_completa.parquet')

# Salvando o GeoDataFrame de input com as estações, sua classificação de 
# representatividade espacial e o tamanho do buffer
filtered_stations.to_csv(outputs_path / 'rep_espacial.csv')

In [6]:
filtered_stations.columns

Index(['UF', 'CIDADE', 'CD_MUN', 'ID_OEMA', 'ID_MMA', 'ID_MMA_COMPLETO',
       'PROPRIETARIO', 'PROP_ENTIDADE', 'OPERADOR', 'OP_ENTIDADE',
       'FUNCIONAMENTO', 'CATEGORIA', 'METODO', 'CALIBRACAO', 'MARCA', 'MODELO',
       'POLUENTE', 'COD_POLUENTE', 'MOBILIDADE', 'FINALIDADE', 'STATUS',
       'INICIO', 'FIM', 'LATITUDE', 'LONGITUDE', 'MONITORAR', 'FONTE',
       'CERTIFICACAO', 'COD_UF_IBGE', 'ANOS_MONITORADOS', 'BASE_DADOS',
       'ELEVACAO', 'REALOCACAO', 'OBS_CALIBRACAO', 'DADOS_MONITORAMENTO',
       'RECONHECIDA', 'OBS_GERAIS', 'REP_ESPACIAL_DECLARADA', 'OPERACAO',
       'geometry', 'REP_ESPACIAL_NAME', 'osm_id_mais_prox_valida',
       'REP_ESPACIAL'],
      dtype='object')

In [7]:
filtered_stations

,UF,CIDADE,CD_MUN,ID_OEMA,ID_MMA,ID_MMA_COMPLETO,PROPRIETARIO,PROP_ENTIDADE,OPERADOR,OP_ENTIDADE,...,OBS_CALIBRACAO,DADOS_MONITORAMENTO,RECONHECIDA,OBS_GERAIS,REP_ESPACIAL_DECLARADA,OPERACAO,geometry,REP_ESPACIAL_NAME,osm_id_mais_prox_valida,REP_ESPACIAL
0,RR,Boa Vista,1400100,Estação FEMARH,RR0002,RR0002RS007,Nao declarado,Nao declarado,Nao declarado,Nao declarado,...,Nao declarado,Nao declarado,None,Nao declarado,Nao declarado,Nao declarado,POINT (-60.70236 2.95196),Bairro,408899273,4000
1,RR,Boa Vista,1400100,Estação Fazenda Carolina,RR0001,RR0001RS007,Nao declarado,Nao declarado,Nao declarado,Nao declarado,...,Nao declarado,Nao declarado,None,Nao declarado,Nao declarado,Nao declarado,POINT (-60.6643 2.82962),Bairro,368956930,4000
2,CE,São Gonçalo do Amarante,2312403,UM Parada Pecem,CE0003,CE0003ND007,Nao declarado,Nao declarado,Nao declarado,Nao declarado,...,Nao declarado,Nao declarado,None,Nao declarado,Nao declarado,Nao declarado,POINT (-38.8842 -3.5684),Bairro,450518017,4000
3,CE,São Gonçalo do Amarante,2312403,CIPP,CE0001,CE0001ND007,Nao declarado,Nao declarado,Nao declarado,Nao declarado,...,Nao declarado,Nao declarado,None,Nao declarado,Nao declarado,Nao declarado,POINT (-38.83664 -3.5558),Micro,240735422,100
4,ES,Anchieta,3200409,EMQAR SUL 03 - Guanabara,ES0014,ES0014RA007,Samarco Mineração S.A.,Privada,JCTM/Acoem,Privada,...,Nao declarado,Nao declarado,None,Nao declarado,Nao declarado,Nao declarado,POINT (-40.62637 -20.82137),Bairro,210067714,4000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
204,PE,Ipojuca,2607208,RNEST ESCOLA IPOJUCA,PE0002,PE0002ND003,Nao declarado,Nao declarado,Nao declarado,Nao declarado,...,Nao declarado,Nao declarado,None,Nao declarado,Nao declarado,Nao declarado,POINT (-35.01262 -8.43811),Bairro,1302726895,4000
205,PE,Ipojuca,2607208,RNEST IFPE,PE0004,PE0004ND003,Nao declarado,Nao declarado,Nao declarado,Nao declarado,...,Nao declarado,Nao declarado,None,Nao declarado,Nao declarado,Nao declarado,POINT (-35.04156 -8.38485),Bairro,542946280,4000
206,PE,Cabo de Santo Agostinho,2602902,RNEST CPRH,PE0003,PE0003ND003,Nao declarado,Nao declarado,Nao declarado,Nao declarado,...,Nao declarado,Nao declarado,None,Nao declarado,Nao declarado,Nao declarado,POINT (-35.02217 -8.33763),Bairro,386100723,4000
207,AL,Maceio,2704302,Braskem,AL0002,AL0002RA003,Nao declarado,Privada,Nao declarado,Privada,...,Nao declarado,Nao declarado,None,Nao declarado,Nao declarado,Nao declarado,POINT (-35.74143 -9.6235),Bairro,37061736,4000
